# 03_exploratory_analyses

Dieses Notebook dokumentiert die explorativen Zusatzanalysen der Meerjungfrauenmärchen-Studie auf Basis der bereinigten und strukturierten Datensätze.

Im Mittelpunkt stehen:

- explorative Analysen der Bewertungsdimensionen
- itembezogene Zusatzanalysen
- explorative Auswertung der Vergleichsfragen
- Zusammenhänge zwischen Print Exposure und Bewertung bzw. Erkennungsleistung

Die in diesem Notebook enthaltenen Auswertungen sind **explorativ** und dienen der vertieften Interpretation der zentralen Ergebnisse.

In [1]:
#Bibliotheken laden

import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
import matplotlib.pyplot as plt

## 1. Bereinigte Datensätze laden

Dieses Notebook arbeitet ausschließlich mit den bereits aufbereiteten Analyse-Dateien.

In [2]:
#from google.colab import drive
#drive.mount('/content/drive')

base_dir = Path("") #Pfad eingeben
data_dir = base_dir / "data" / "processed"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
#Einlesen

df_master = pd.read_excel(data_dir / "master_clean_renamed.xlsx")
ratings_long = pd.read_excel(data_dir / "ratings_long.xlsx")
compare_long = pd.read_excel(data_dir / "compare_long.xlsx")
estimation_long = pd.read_excel(data_dir / "estimation_long.xlsx")

# Nur laden, wenn die Dateien vorhanden und datenschutzrechtlich freigegeben sind
open_item9_path = data_dir / "open_item9_long.xlsx"
open_compare2_path = data_dir / "open_compare2_long.xlsx"

open_item9_long = pd.read_excel(open_item9_path) if open_item9_path.exists() else None
open_compare2_long = pd.read_excel(open_compare2_path) if open_compare2_path.exists() else None

print("master:", df_master.shape)
print("ratings_long:", ratings_long.shape)
print("compare_long:", compare_long.shape)
print("estimation_long:", estimation_long.shape)
if open_item9_long is not None:
    print("open_item9_long:", open_item9_long.shape)
if open_compare2_long is not None:
    print("open_compare2_long:", open_compare2_long.shape)


master: (138, 610)
ratings_long: (10585, 10)
compare_long: (2592, 9)
estimation_long: (552, 9)
open_item9_long: (373, 10)
open_compare2_long: (384, 9)


## 2. Vorbereitungen

Für die Analysen werden die zentralen Bewertungsdimensionen nach Kienecker sowie weitere Hilfsvariablen definiert.

In [4]:
# Nur geschlossene Kernitems 1-8 für die Qualitätsdimensionen
ratings_core = ratings_long[ratings_long["item_no"].between(1, 8)].copy()

# Bewertungsdimensionen zuordnen
appetenz_items = [1, 2, 4, 5]
leistung_items = [3, 6]
akzeptanz_items = [7]
bedeutung_items = [8]

dim_map = {}
for item in appetenz_items:
    dim_map[item] = "Appetenz"
for item in leistung_items:
    dim_map[item] = "Leistung"
for item in akzeptanz_items:
    dim_map[item] = "Akzeptanz"
for item in bedeutung_items:
    dim_map[item] = "Bedeutung"

ratings_core["dimension"] = ratings_core["item_no"].map(dim_map)

# Hilfslabels für Einzelitems
item_labels = {
    1: "Gefallen",
    2: "Neugier_auf_Handlung",
    3: "Verstaendlichkeit",
    4: "Vorstellbarkeit",
    5: "Emotionale_Wirkung",
    6: "Sprachliche_Kunstfertigkeit",
    7: "Maerchenhaftigkeit",
    8: "Bedeutungszuschreibung",
    9: "Offene_Antwort",
    10: "Zusatzitem_10",
    11: "Zusatzitem_11",
}

ratings_long["item_label"] = ratings_long["item_no"].map(item_labels)
ratings_core["item_label"] = ratings_core["item_no"].map(item_labels)

# Vergleichsfragenlabels
compare_labels = {
    1: "Gesamtpraeferenz",
    2: "Offene_Begruendung",
    3: "Stilistische_Aehnlichkeit",
    4: "Vergleichsfrage_4",
    5: "Vergleichsfrage_5",
}
compare_long["question_label"] = compare_long["question_no"].map(compare_labels)

## 3. Analyse der Bewertungsdimensionen

Hier wird geprüft, wie sich die Bewertungsdimensionen Appetenz, Leistung, Akzeptanz und Bedeutungszuschreibung zwischen Gruppen und Texttypen verhalten.

In [5]:
# Dimensionen pro Person x Texttyp

dimension_person = (
    ratings_core
    .groupby(["participant_id", "group", "text_type", "dimension"], as_index=False)["response_num"]
    .mean()
    .rename(columns={"response_num": "dimension_score"})
)

dimension_person.head()

,participant_id,group,text_type,dimension,dimension_score
0,4,B,ai,Akzeptanz,5.2500
1,4,B,ai,Appetenz,5.9375
2,4,B,ai,Bedeutung,5.2500
3,4,B,ai,Leistung,5.7500
4,4,B,human,Akzeptanz,5.2500


In [6]:
# Deskriptive Mittelwerte pro Gruppe pro Texttyp

dimension_summary = (
    dimension_person
    .groupby(["group", "text_type", "dimension"], as_index=False)["dimension_score"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

dimension_summary

,index,group,text_type,dimension,mean,std,count
0,0,A,ai,Akzeptanz,4.974638,1.004718,69
1,1,A,ai,Appetenz,5.230978,0.893406,69
2,2,A,ai,Bedeutung,4.467391,1.131040,69
3,3,A,ai,Leistung,5.425725,0.699698,69
4,4,A,human,Akzeptanz,4.905797,0.935514,69
5,5,A,human,Appetenz,4.557065,0.852369,69
6,6,A,human,Bedeutung,4.184783,0.930145,69
7,7,A,human,Leistung,5.014493,0.834553,69
8,8,B,ai,Akzeptanz,4.478261,1.078482,69
9,9,B,ai,Appetenz,4.495471,1.109644,69


In [7]:
# Delta pro Dimension

dimension_wide = (
    dimension_person
    .pivot(index=["participant_id", "group", "dimension"], columns="text_type", values="dimension_score")
    .reset_index()
)

dimension_wide["DeltaDimension"] = dimension_wide["human"] - dimension_wide["ai"]

dimension_delta_summary = (
    dimension_wide
    .groupby(["group", "dimension"], as_index=False)["DeltaDimension"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

dimension_delta_summary

,index,group,dimension,mean,std,count
0,0,A,Akzeptanz,-0.068841,1.267440,69
1,1,A,Appetenz,-0.673913,1.014026,69
2,2,A,Bedeutung,-0.282609,1.278292,69
3,3,A,Leistung,-0.411232,0.909029,69
4,4,B,Akzeptanz,0.952899,1.427330,69
5,5,B,Appetenz,0.516304,1.334448,69
6,6,B,Bedeutung,1.050725,1.667476,69
7,7,B,Leistung,0.507246,0.933172,69


In [8]:
# Welch-t-Tests der Deltas zwischen Gruppe A und Gruppe B

results_dim = []

for dim in sorted(dimension_wide["dimension"].dropna().unique()):
    sub = dimension_wide[dimension_wide["dimension"] == dim]
    delta_A = sub.loc[sub["group"] == "A", "DeltaDimension"].dropna()
    delta_B = sub.loc[sub["group"] == "B", "DeltaDimension"].dropna()

    t_stat, p_val = stats.ttest_ind(delta_A, delta_B, equal_var=False)

    results_dim.append({
        "dimension": dim,
        "mean_delta_A": delta_A.mean(),
        "mean_delta_B": delta_B.mean(),
        "t_welch": t_stat,
        "p": p_val
    })

results_dim_df = pd.DataFrame(results_dim)
results_dim_df

,dimension,mean_delta_A,mean_delta_B,t_welch,p
0,Akzeptanz,-0.068841,0.952899,-4.446259,1.815706e-05
1,Appetenz,-0.673913,0.516304,-5.898955,3.117641e-08
2,Bedeutung,-0.282609,1.050725,-5.271353,5.617248e-07
3,Leistung,-0.411232,0.507246,-5.856445,3.384970e-08


## 4. Explorative Analyse der Einzelitems

Hier wird geprüft, welche einzelnen Bewertungsitems besonders stark zwischen den Gruppen differieren.

In [11]:
# Itemmittelwerte pro Person x Teyttyp

item_person = (
    ratings_core
    .groupby(["participant_id", "group", "text_type", "item_no", "item_label"], as_index=False)["response_num"]
    .mean()
    .rename(columns={"response_num": "item_score"})
)

item_person.head()

,participant_id,group,text_type,item_no,item_label,item_score
0,4,B,ai,1,Gefallen,6.25
1,4,B,ai,2,Neugier_auf_Handlung,6.00
2,4,B,ai,3,Verstaendlichkeit,6.75
3,4,B,ai,4,Vorstellbarkeit,6.25
4,4,B,ai,5,Emotionale_Wirkung,5.25


In [12]:
# Delta pro Item

item_wide = (
    item_person
    .pivot(index=["participant_id", "group", "item_no", "item_label"], columns="text_type", values="item_score")
    .reset_index()
)

item_wide["DeltaItem"] = item_wide["human"] - item_wide["ai"]

item_delta_summary = (
    item_wide
    .groupby(["group", "item_no", "item_label"], as_index=False)["DeltaItem"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

item_delta_summary

,index,group,item_no,item_label,mean,std,count
0,0,A,1,Gefallen,-0.757246,1.248507,69
1,1,A,2,Neugier_auf_Handlung,-0.782609,1.303215,69
2,2,A,3,Verstaendlichkeit,-1.112319,1.024100,69
3,3,A,4,Vorstellbarkeit,-0.536232,0.968507,69
4,4,A,5,Emotionale_Wirkung,-0.619565,1.149335,69
5,5,A,6,Sprachliche_Kunstfertigkeit,0.289855,1.309344,69
6,6,A,7,Maerchenhaftigkeit,-0.068841,1.267440,69
7,7,A,8,Bedeutungszuschreibung,-0.282609,1.278292,69
8,8,B,1,Gefallen,0.605072,1.526365,69
9,9,B,2,Neugier_auf_Handlung,0.485507,1.603583,69


In [13]:
# Welch-t-Tests pro Item

results_item = []

for item_no in sorted(item_wide["item_no"].dropna().unique()):
    sub = item_wide[item_wide["item_no"] == item_no]
    delta_A = sub.loc[sub["group"] == "A", "DeltaItem"].dropna()
    delta_B = sub.loc[sub["group"] == "B", "DeltaItem"].dropna()

    t_stat, p_val = stats.ttest_ind(delta_A, delta_B, equal_var=False)

    results_item.append({
        "item_no": item_no,
        "item_label": sub["item_label"].iloc[0],
        "mean_delta_A": delta_A.mean(),
        "mean_delta_B": delta_B.mean(),
        "difference_B_minus_A": delta_B.mean() - delta_A.mean(),
        "t_welch": t_stat,
        "p": p_val
    })

results_item_df = pd.DataFrame(results_item).sort_values("difference_B_minus_A", ascending=False)
results_item_df

,item_no,item_label,mean_delta_A,mean_delta_B,difference_B_minus_A,t_welch,p
5,6,Sprachliche_Kunstfertigkeit,0.289855,2.028986,1.739130,-7.350665,1.736454e-11
0,1,Gefallen,-0.757246,0.605072,1.362319,-5.738639,6.315825e-08
7,8,Bedeutungszuschreibung,-0.282609,1.050725,1.333333,-5.271353,5.617248e-07
4,5,Emotionale_Wirkung,-0.619565,0.695652,1.315217,-5.890188,3.152709e-08
1,2,Neugier_auf_Handlung,-0.782609,0.485507,1.268116,-5.097736,1.179662e-06
6,7,Maerchenhaftigkeit,-0.068841,0.952899,1.021739,-4.446259,1.815706e-05
3,4,Vorstellbarkeit,-0.536232,0.278986,0.815217,-4.367419,2.547947e-05
2,3,Verstaendlichkeit,-1.112319,-1.014493,0.097826,-0.570053,5.695835e-01


## 5. Textpaarspezifische Zusatzitems (Item 10 / 11)

Zusätzlich zu den Kernitems werden die textspezifischen Zusatzitems explorativ untersucht.

In [14]:
# Zusatzitems filtern

ratings_extra = ratings_long[ratings_long["item_no"].isin([10, 11])].copy()
ratings_extra["response_num"] = pd.to_numeric(ratings_extra["response_num"], errors="coerce")

extra_person = (
    ratings_extra
    .groupby(["participant_id", "group", "pair_id", "text_type", "item_no"], as_index=False)["response_num"]
    .mean()
    .rename(columns={"response_num": "extra_score"})
)

extra_person.head()

,participant_id,group,pair_id,text_type,item_no,extra_score
0,4,B,1,ai,10,7.0
1,4,B,1,human,10,5.0
2,4,B,2,ai,10,5.0
3,4,B,2,ai,11,6.0
4,4,B,2,human,10,6.0


In [15]:
# Zusatzitems summarisch

extra_summary = (
    extra_person
    .groupby(["group", "pair_id", "text_type", "item_no"], as_index=False)["extra_score"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

extra_summary

,index,group,pair_id,text_type,item_no,mean,std,count
0,0,A,1,ai,10,5.246377,1.439013,69
1,1,A,1,human,10,4.695652,1.665430,69
2,2,A,2,ai,10,4.521739,1.471381,69
3,3,A,2,ai,11,5.188406,1.275047,69
4,4,A,2,human,10,5.130435,1.552162,69
5,5,A,2,human,11,4.913043,1.704140,69
6,6,A,3,ai,10,5.304348,1.427713,69
7,7,A,3,human,10,5.173913,1.543073,69
8,8,A,4,ai,10,5.637681,1.224397,69
9,9,A,4,human,10,4.840580,1.491379,69


In [42]:
# Basisindex: nur Items 1–8
base_person = (
    ratings_long[ratings_long["item_no"].isin(range(1, 9))].copy()
    .assign(response_num=lambda d: pd.to_numeric(d["response_num"], errors="coerce"))
    .groupby(["participant_id", "group", "pair_id", "text_type"], as_index=False)["response_num"]
    .mean()
    .rename(columns={"response_num": "score"})
)

base_wide = (
    base_person
    .groupby(["participant_id", "group", "text_type"], as_index=False)["score"]
    .mean()
    .pivot(index=["participant_id", "group"], columns="text_type", values="score")
    .reset_index()
)
base_wide["Delta"] = base_wide["human"] - base_wide["ai"]

# Erweiterter Index: Items 1–8 + Zusatzitems 10/11
ext_person = (
    ratings_long[ratings_long["item_no"].isin(list(range(1, 9)) + [10, 11])].copy()
    .assign(response_num=lambda d: pd.to_numeric(d["response_num"], errors="coerce"))
    .groupby(["participant_id", "group", "pair_id", "text_type"], as_index=False)["response_num"]
    .mean()
    .rename(columns={"response_num": "score"})
)

ext_wide = (
    ext_person
    .groupby(["participant_id", "group", "text_type"], as_index=False)["score"]
    .mean()
    .pivot(index=["participant_id", "group"], columns="text_type", values="score")
    .reset_index()
)
ext_wide["Delta"] = ext_wide["human"] - ext_wide["ai"]

print("Nur Items 1–8:")
print(base_wide.groupby("group")["Delta"].mean())

print("\nItems 1–8 + 10/11:")
print(ext_wide.groupby("group")["Delta"].mean())

Nur Items 1–8:
group
A   -0.483696
B    0.635417
Name: Delta, dtype: float64

Items 1–8 + 10/11:
group
A   -0.455314
B    0.651731
Name: Delta, dtype: float64


## 6. Explorative Analyse der Vergleichsfragen

Hier werden die direkten Vergleichsfragen zu den Textpaaren betrachtet.

In [16]:
compare_pref = compare_long[compare_long["question_no"].isin([1, 4, 5])].copy()
compare_pref["response_num"] = pd.to_numeric(compare_pref["response_num"], errors="coerce")

compare_pref.head()

,participant_id,group,column_name,pair_id,question_no,raw_response,response_num,response_text,preference_target,question_label
0,8,A,VA131. Welcher der beiden Textausschnitte wirk...,1,1,4 – beide Textausschnitte gleich stark,4.0,NaN,neutral,Gesamtpraeferenz
1,9,A,VA131. Welcher der beiden Textausschnitte wirk...,1,1,7 – nur Textausschnitt 2,7.0,NaN,ai,Gesamtpraeferenz
2,13,A,VA131. Welcher der beiden Textausschnitte wirk...,1,1,2 – hauptsächlich Textausschnitt 1,2.0,NaN,human,Gesamtpraeferenz
3,15,A,VA131. Welcher der beiden Textausschnitte wirk...,1,1,4 – beide Textausschnitte gleich stark,4.0,NaN,neutral,Gesamtpraeferenz
4,16,A,VA131. Welcher der beiden Textausschnitte wirk...,1,1,4 – beide Textausschnitte gleich stark,4.0,NaN,neutral,Gesamtpraeferenz


In [17]:
# Gesamtpräferenzindex bilden

pref_overall = (
    compare_pref
    .groupby(["participant_id", "group", "pair_id"], as_index=False)["response_num"]
    .mean()
    .rename(columns={"response_num": "PrefOverall_raw"})
)

# Umkodierung auf Skala mit Mittelpunkt 4 -> neutral
# Optional: human/ai-orientierte Differenzskala
def recode_pref_centered(row):
    val = row["PrefOverall_raw"]
    if pd.isna(val):
        return np.nan

    # Paar 1 und 4: Text1=human, Text2=ai
    if row["pair_id"] in [1, 4]:
        return 4 - val  # positiv = human, negativ = ai
    # Paar 2 und 3: Text1=ai, Text2=human
    elif row["pair_id"] in [2, 3]:
        return val - 4  # positiv = human, negativ = ai
    return np.nan

pref_overall["PrefOverall_centered"] = pref_overall.apply(recode_pref_centered, axis=1)
pref_overall.head()

,participant_id,group,pair_id,PrefOverall_raw,PrefOverall_centered
0,4,B,1,5.000000,-1.000000
1,4,B,2,4.000000,0.000000
2,4,B,3,7.000000,3.000000
3,4,B,4,2.000000,2.000000
4,8,A,1,4.666667,-0.666667


In [18]:
# Präferent nach Gruppe

pref_group = (
    pref_overall
    .groupby("group")["PrefOverall_centered"]
    .agg(["mean", "std", "count"])
)

pref_group

,mean,std,count
group,,,
A,-0.258454,1.300205,276
B,0.667874,1.363447,276


In [44]:
# Welch-t-Test Präferenz A vs B

pref_person = (
    pref_overall
    .groupby(["participant_id", "group"], as_index=False)["PrefOverall_centered"]
    .mean()
)

pref_A = pref_person.loc[pref_person["group"] == "A", "PrefOverall_centered"].dropna()
pref_B = pref_person.loc[pref_person["group"] == "B", "PrefOverall_centered"].dropna()

t_pref, p_pref = stats.ttest_ind(pref_A, pref_B, equal_var=False)

mean_A = pref_A.mean()
sd_A = pref_A.std(ddof=1)
mean_B = pref_B.mean()
sd_B = pref_B.std(ddof=1)

n1, n2 = len(pref_A), len(pref_B)

pooled_sd = np.sqrt(((n1 - 1) * sd_A**2 + (n2 - 1) * sd_B**2) / (n1 + n2 - 2))
d_pref = (mean_B - mean_A) / pooled_sd

df_welch = ((sd_A**2 / n1 + sd_B**2 / n2) ** 2) / (
    ((sd_A**2 / n1) ** 2) / (n1 - 1) +
    ((sd_B**2 / n2) ** 2) / (n2 - 1)
)

print("A:", mean_A, sd_A, "n =", n1)
print("B:", mean_B, sd_B, "n =", n2)
print("Welch-t:", t_pref)
print("df =", df_welch)
print("p:", p_pref)
print("Cohen's d =", d_pref)

A: -0.2584541062801932 0.826707102843688 n = 69
B: 0.6678743961352656 0.8548081944805755 n = 69
Welch-t: -6.4705785045741155
df = 135.84832414026425
p: 1.6363027039713175e-09
Cohen's d = 1.1016244419611785


In [20]:
# Vergleichsfrage 3 stilitsiche Ähnlichkeit

compare_sim = compare_long[compare_long["question_no"] == 3].copy()
compare_sim["response_num"] = pd.to_numeric(compare_sim["response_num"], errors="coerce")

sim_group = compare_sim.groupby("group")["response_num"].agg(["mean", "std", "count"])
sim_group

,mean,std,count
group,,,
A,3.043478,1.692205,276
B,2.489130,1.386909,276


In [21]:
# Welch-t-Test Ähnlichkeit

sim_A = compare_sim.loc[compare_sim["group"] == "A", "response_num"].dropna()
sim_B = compare_sim.loc[compare_sim["group"] == "B", "response_num"].dropna()

t_sim, p_sim = stats.ttest_ind(sim_A, sim_B, equal_var=False)

print("Stilistische Ähnlichkeit A vs B")
print("A:", sim_A.mean(), sim_A.std(ddof=1))
print("B:", sim_B.mean(), sim_B.std(ddof=1))
print("Welch-t:", t_sim)
print("p:", p_sim)

Stilistische Ähnlichkeit A vs B
A: 3.0434782608695654 1.692204867104734
B: 2.489130434782609 1.3869094370438395
Welch-t: 4.209221397389011
p: 3.0104464621778166e-05


## 7. Explorative Zusammenhänge mit Print Exposure (ART)

Hier wird geprüft, ob die ART-Werte mit Bewertung und Erkennungsleistung zusammenhängen.

In [22]:
# DeltaQuality vorbereiten

quality_person = (
    ratings_core
    .groupby(["participant_id", "group", "text_type"], as_index=False)["response_num"]
    .mean()
    .rename(columns={"response_num": "QualityIndex"})
)

quality_wide = (
    quality_person
    .pivot(index=["participant_id", "group"], columns="text_type", values="QualityIndex")
    .reset_index()
)

quality_wide["DeltaQuality"] = quality_wide["human"] - quality_wide["ai"]
quality_wide.head()

text_type,participant_id,group,ai,human,DeltaQuality
0,4,B,5.71875,5.00000,-0.71875
1,8,A,4.84375,3.78125,-1.06250
2,9,A,4.21875,3.37500,-0.84375
3,13,A,4.75000,4.90625,0.15625
4,14,B,3.87500,5.87500,2.00000


In [23]:
# Accuracy pro Person

accuracy_person = (
    estimation_long
    .groupby(["participant_id", "group"], as_index=False)["correct"]
    .mean()
    .rename(columns={"correct": "Accuracy"})
)

accuracy_person.head()

,participant_id,group,Accuracy
0,4,B,0.75
1,8,A,0.00
2,9,A,0.50
3,13,A,0.75
4,14,B,1.00


In [24]:
# Mit df_master zusammenführen

analysis_person = (
    df_master[["participant_id", "group", "ART_rate", "study_or_job_raw", "reading_frequency_raw", "age", "gender"]]
    .merge(quality_wide, on=["participant_id", "group"], how="left")
    .merge(accuracy_person, on=["participant_id", "group"], how="left")
)

analysis_person.head()

,participant_id,group,ART_rate,study_or_job_raw,reading_frequency_raw,age,gender,ai,human,DeltaQuality,Accuracy
0,4,B,0.520000,Intelligent Systems Engineering,Etwa einmal pro Monat,35-44 Jahre,Weiblich,5.71875,5.00000,-0.71875,0.75
1,8,A,0.413333,Digital Healthcare Management,Seltener als einmal im Monat (z.B. nur im Urlaub),25-34 Jahre,Männlich,4.84375,3.78125,-1.06250,0.00
2,9,A,0.466667,Pharmazie,Etwa einmal pro Monat,25-34 Jahre,Weiblich,4.21875,3.37500,-0.84375,0.50
3,13,A,0.626667,Datenanalystin,Mehrmals pro Woche,25-34 Jahre,Weiblich,4.75000,4.90625,0.15625,0.75
4,14,B,0.680000,BSc. Biologie,Seltener als einmal im Monat (z.B. nur im Urlaub),25-34 Jahre,Weiblich,3.87500,5.87500,2.00000,1.00


In [25]:
# Korrelationen mit ART

def pearson_and_spearman(x, y, label):
    sub = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(sub) < 3:
        return {"label": label, "n": len(sub), "pearson_r": np.nan, "pearson_p": np.nan,
                "spearman_r": np.nan, "spearman_p": np.nan}

    pr, pp = stats.pearsonr(sub["x"], sub["y"])
    sr, sp = stats.spearmanr(sub["x"], sub["y"])
    return {
        "label": label,
        "n": len(sub),
        "pearson_r": pr,
        "pearson_p": pp,
        "spearman_r": sr,
        "spearman_p": sp
    }

corr_results = []
corr_results.append(pearson_and_spearman(analysis_person["ART_rate"], analysis_person["human"], "ART vs Human_mean"))
corr_results.append(pearson_and_spearman(analysis_person["ART_rate"], analysis_person["ai"], "ART vs AI_mean"))
corr_results.append(pearson_and_spearman(analysis_person["ART_rate"], analysis_person["DeltaQuality"], "ART vs DeltaQuality"))
corr_results.append(pearson_and_spearman(analysis_person["ART_rate"], analysis_person["Accuracy"], "ART vs Accuracy"))

pd.DataFrame(corr_results)

,label,n,pearson_r,pearson_p,spearman_r,spearman_p
0,ART vs Human_mean,138,0.181992,0.032650,0.156732,0.066388
1,ART vs AI_mean,138,-0.089499,0.296526,-0.066337,0.439493
2,ART vs DeltaQuality,138,0.192986,0.023341,0.178857,0.035822
3,ART vs Accuracy,138,0.137254,0.108426,0.139280,0.103262


## 8. Explorative Analyse nach Berufsfeld

Die Freitextangaben zu Studium/Beruf werden zu Berufsfeldkategorien zusammengefasst und explorativ mit Qualitätsurteilen und Accuracy in Beziehung gesetzt.

In [35]:
def contains_pattern(text, patterns):
    return any(re.search(p, text) for p in patterns)

def map_job_field(text):
    if pd.isna(text):
        return "Andere/keine Angabe"
    t = str(text).strip().lower()
    t = (
        t.replace("ä", "ae")
         .replace("ö", "oe")
         .replace("ü", "ue")
         .replace("ß", "ss")
    )


    if t == "":
        return "Andere/keine Angabe"

    if contains_pattern(t, [
        r"lehramt", r"lehr", r"erzieh", r"paedagog", r"sonderpaed",
        r"heilpaedagog", r"grundschule", r"kinderbetreuung"
    ]):
        return "Bildung / Pädagogik"

    if contains_pattern(t, [
        r"germanistik", r"literatur", r"philolog", r"anglist",
        r"kulturwissenschaft", r"philosophie", r"musikwissenschaft",
        r"histor", r"archiv", r"bibliothek", r"buchhaend", r"verlag",
        r"autor", r"schriftstell", r"museum",
        r"wissenschaftlicher mitarbeiter.*histor",
        r"wissenschaftlicher mitarbeiter.*bibliothek"
    ]):
        return "Literatur & Kulturbereich"

    if contains_pattern(t, [
        r"digital humanities", r"\bdh\b"
    ]):
        return "Digital Humanities"

    if contains_pattern(t, [
        r"informat", r"software", r"\bit\b", r"machine learning",
        r"\bml\b", r"nlp", r"datenanalyst", r"data",
        r"intelligent systems", r"fachinformat", r"computerlingu",
        r"entwickler", r"programm"
    ]):
        return "IT / KI / Technik"

    if contains_pattern(t, [
        r"biolog", r"pharma", r"medizin", r"pflege", r"logopaed",
        r"arzt", r"chem", r"physio", r"bauchem",
        r"bewegungswissenschaft", r"kranken", r"notfall"
    ]):
        return "Naturwissenschaft & Medizin"

    if contains_pattern(t, [
        r"psycholog", r"soziolog", r"sozialarbeit", r"sozialpaed",
        r"kriminolog", r"sozial", r"polit"
    ]):
        return "Sozialwissenschaften"

    if contains_pattern(t, [
        r"wirtschaft", r"\bbwl\b", r"\bvwl\b", r"management",
        r"consult", r"einkauf", r"buchhalt", r"steuer", r"notar",
        r"verwaltung", r"oeffentlich", r"vertrieb", r"\bhr\b",
        r"industriekaufmann", r"betriebswirt", r"e-commerce",
        r"projektkoordination", r"referent", r"rechtsreferendar",
        r"sekretaer", r"bank"
    ]):
        return "Wirtschaft / Verwaltung"

    if contains_pattern(t, [
        r"medien", r"kommunikation", r"\bpr\b", r"marketing",
        r"campaign", r"theater", r"design", r"buehne"
    ]):
        return "Medien & Kreativ"

    if contains_pattern(t, [
        r"gastr", r"schrein", r"service", r"techn zeichn", r"seifen"
    ]):
        return "Handwerk / Service"

    return "Andere/keine Angabe"


analysis_person["job_field"] = analysis_person["study_or_job_raw"].apply(map_job_field)
analysis_person["job_field"].value_counts()

,count
job_field,
Andere/keine Angabe,28
Wirtschaft / Verwaltung,26
Naturwissenschaft & Medizin,19
Bildung / Pädagogik,15
Literatur & Kulturbereich,13
Sozialwissenschaften,13
IT / KI / Technik,8
Digital Humanities,8
Medien & Kreativ,4


In [37]:
# DeltaQuality nach Berufsfeld

job_quality = (
    analysis_person
    .groupby("job_field", as_index=False)["DeltaQuality"]
    .agg(["mean", "std", "count"])
    .reset_index()
    .sort_values("mean", ascending=False)
)

job_quality

,index,job_field,mean,std,count
4,4,IT / KI / Technik,0.363281,1.289762,8
3,3,Handwerk / Service,0.343750,1.019663,4
2,2,Digital Humanities,0.296875,1.334217,8
5,5,Literatur & Kulturbereich,0.201923,1.315820,13
1,1,Bildung / Pädagogik,0.118750,0.957320,15
0,0,Andere/keine Angabe,0.084821,1.030140,28
9,9,Wirtschaft / Verwaltung,0.081731,1.211729,26
8,8,Sozialwissenschaften,-0.122596,1.491374,13
7,7,Naturwissenschaft & Medizin,-0.134868,1.366334,19
6,6,Medien & Kreativ,-0.234375,0.665608,4


In [38]:
# Accuracy nach Berufsfeld

job_acc = (
    analysis_person
    .groupby("job_field", as_index=False)["Accuracy"]
    .agg(["mean", "std", "count"])
    .reset_index()
    .sort_values("mean", ascending=False)
)

job_acc

,index,job_field,mean,std,count
2,2,Digital Humanities,0.781250,0.247758,8
3,3,Handwerk / Service,0.687500,0.314576,4
4,4,IT / KI / Technik,0.687500,0.320435,8
5,5,Literatur & Kulturbereich,0.673077,0.328897,13
7,7,Naturwissenschaft & Medizin,0.657895,0.325006,19
8,8,Sozialwissenschaften,0.634615,0.262508,13
9,9,Wirtschaft / Verwaltung,0.625000,0.340955,26
0,0,Andere/keine Angabe,0.598214,0.306860,28
1,1,Bildung / Pädagogik,0.583333,0.322749,15
6,6,Medien & Kreativ,0.375000,0.250000,4


## 9. Zusammenfassung der explorativen Analysen

Die explorativen Analysen dienen der vertieften Einordnung der zentralen Befunde. Sie zeigen unter anderem:

- konsistente Gruppenunterschiede in den Bewertungsdimensionen
- Unterschiede in der Sensitivität einzelner Items
- ein ähnliches Muster zwischen Einzelbewertungen und direkten Vergleichsfragen
- leichte Zusammenhänge zwischen Print Exposure und Qualitätsurteilen

Diese Analysen sind explorativ und werden entsprechend vorsichtig interpretiert.